# Model 2 — Conditional GAN Date Generator
Run on a **T4 GPU** Colab: Runtime → Change runtime type → T4 GPU

In [ ]:
# ── 1. Clone & install ────────────────────────────────────────────────────────
REPO = "https://github.com/SalmaSherif7070/Conditional-Date-Generation-Using-Deep-Generative-Models"
!git clone {REPO} repo
%cd repo
!pip install -q -r requirements.txt

In [ ]:
# ── 2. GPU check ─────────────────────────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('GPU    :', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NOT available — set Runtime → T4 GPU')

In [ ]:
# ── 3. Create __init__.py files ───────────────────────────────────────────────
import os
for pkg in ['src', 'src/model_1', 'src/model_2', 'src/model_3', 'src/model_4']:
    os.makedirs(pkg, exist_ok=True)
    p = os.path.join(pkg, '__init__.py')
    if not os.path.exists(p):
        open(p, 'w').close()
print('✓ __init__.py ready')

In [ ]:
# ── 4. Write src/model_2/visualization.py ────────────────────────────────────
viz_code = '''
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

STYLE  = {"figure.facecolor": "white", "axes.spines.top": False, "axes.spines.right": False}
C_TR   = "#4C72B0"
C_VAL  = "#DD8452"
COLORS = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974"]

def _save(fig, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")

def plot_loss_curves(history, save_dir):
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(history["epochs"], history["g_loss"], lw=2, color=C_TR,  label="G Loss")
        ax.plot(history["epochs"], history["d_loss"], lw=2, color=C_VAL, label="D Loss", ls="--")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.set_title("Model 2 – GAN Loss (G vs D)"); ax.legend(); ax.grid(alpha=0.3)
        _save(fig, os.path.join(save_dir, "model2_loss_curves.png"))

def plot_loss_log_scale(history, save_dir):
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.semilogy(history["epochs"], [abs(v) for v in history["g_loss"]], lw=2, color=C_TR,  label="|G Loss|")
        ax.semilogy(history["epochs"], [abs(v) for v in history["d_loss"]], lw=2, color=C_VAL, label="|D Loss|", ls="--")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss (log)")
        ax.set_title("Model 2 – Loss Log Scale"); ax.legend(); ax.grid(alpha=0.3, which="both")
        _save(fig, os.path.join(save_dir, "model2_loss_log.png"))

def plot_condition_breakdown(metrics, save_dir):
    labels = ["Day of Week", "Month", "Leap Year", "Decade", "All (CSR)"]
    values = [metrics["dow_acc"], metrics["mon_acc"], metrics["leap_acc"],
              metrics["decade_acc"], metrics["csr"]]
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots(figsize=(8, 5))
        bars = ax.bar(labels, values, color=COLORS, edgecolor="white")
        ax.set_ylabel("Accuracy"); ax.set_title("Model 2 – Per-Condition Accuracy")
        ax.set_ylim(0, 1.15)
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.02,
                    f"{val:.1%}", ha="center", va="bottom", fontsize=10)
        ax.grid(axis="y", alpha=0.3)
        _save(fig, os.path.join(save_dir, "model2_condition_breakdown.png"))
'''
with open('src/model_2/visualization.py', 'w') as f:
    f.write(viz_code.strip())
print('✓ src/model_2/visualization.py written')

In [ ]:
# ── 5. Patch config.py — 100 epochs, flat output paths ───────────────────────
config_src = '''
from dataclasses import dataclass

@dataclass
class ModelConfig:
    cond_dim: int = 128
    max_decade: int = 300

@dataclass
class TrainConfig:
    n_epochs: int = 100
    batch_size: int = 256
    lr: float = 2e-3
    val_split: float = 0.2
    seed: int = 42

@dataclass
class Model2Config:
    cond_dim: int = 128
    max_decade: int = 300
    z_dim: int = 64

@dataclass
class Train2Config:
    n_epochs: int = 100
    batch_size: int = 256
    lr_g: float = 1e-4
    lr_d: float = 4e-4
    n_critic: int = 2
    lambda_gp: float = 10.0
    tau_start: float = 2.0
    tau_end: float = 0.5
    val_split: float = 0.2
    seed: int = 42

@dataclass
class Model3Config:
    cond_dim: int = 128
    max_decade: int = 300
    z_dim: int = 64

@dataclass
class Train3Config:
    n_epochs: int = 100
    batch_size: int = 256
    lr: float = 1e-3
    beta_max: float = 0.5
    beta_warmup_frac: float = 0.5
    val_split: float = 0.2
    seed: int = 42

@dataclass
class Model4Config:
    cond_dim: int = 128
    max_decade: int = 300
    hidden_dim: int = 512
    n_layers: int = 4

@dataclass
class Train4Config:
    n_epochs: int = 100
    batch_size: int = 256
    lr: float = 1e-4
    n_mcmc_steps: int = 60
    mcmc_step_size: float = 0.1
    mcmc_noise: float = 0.005
    replay_buffer_size: int = 10_000
    replay_prob: float = 0.95
    l2_reg: float = 1.0
    grad_clip: float = 1.0
    val_split: float = 0.2
    seed: int = 42

@dataclass
class PathConfig:
    data_path: str = "data/raw/data.txt"
    example_input_path: str = "data/raw/example_input.txt"
    output_dir: str = "output"
    weights_path: str = "output/weights.pt"
    figures_dir: str = "output/figures"

@dataclass
class Path2Config:
    data_path: str = "data/raw/data.txt"
    example_input_path: str = "data/raw/example_input.txt"
    output_dir: str = "output"
    generator_path: str = "output/generator.pt"
    discriminator_path: str = "output/discriminator.pt"
    encoder_path: str = "output/encoder.pt"
    figures_dir: str = "output/figures"

@dataclass
class Path3Config:
    data_path: str = "data/raw/data.txt"
    example_input_path: str = "data/raw/example_input.txt"
    output_dir: str = "output"
    weights_path: str = "output/weights.pt"
    figures_dir: str = "output/figures"

@dataclass
class Path4Config:
    data_path: str = "data/raw/data.txt"
    example_input_path: str = "data/raw/example_input.txt"
    output_dir: str = "output"
    weights_path: str = "output/weights.pt"
    figures_dir: str = "output/figures"
'''
with open('src/config.py', 'w') as f:
    f.write(config_src.strip())
print('✓ src/config.py patched')

In [ ]:
# ── 6. Write self-contained main2.py ─────────────────────────────────────────
# Model 2 has a non-standard evaluate call in the original main.py.
# We fix it and write a clean standalone script.
main2_src = '''
import os, torch
from src.config import Model2Config, Train2Config, Path2Config
from src.data_processing import load_dataset, load_example_input, format_output_line
from src.model_2.model import DateGenerator, ConditionEncoder
from src.model_2.train import train as train_model2
from src.model_2.visualization import plot_loss_curves, plot_loss_log_scale, plot_condition_breakdown
from src.model_4.model import is_leap, days_in_month, day_of_week

def check_conditions(dates, conditions):
    d, m, y = dates[:, 0], dates[:, 1], dates[:, 2]
    dow_c, mon_c, leap_c, dec_c = conditions[:, 0], conditions[:, 1], conditions[:, 2], conditions[:, 3]
    ok_month = (m >= 1) & (m <= 12)
    ok_year  = y > 0
    ok_day   = ok_month & ok_year & (d >= 1) & (d <= days_in_month(m, y))
    dow_ok    = ok_day & (day_of_week(d, m, y) == dow_c)
    mon_ok    = ok_day & ((m - 1) == mon_c)
    leap_ok   = ok_day & (is_leap(y).long() == leap_c)
    decade_ok = ok_day & ((y // 10) == dec_c)
    all_ok    = dow_ok & mon_ok & leap_ok & decade_ok
    return {"csr": all_ok.float().mean().item(),
            "dow_acc": dow_ok.float().mean().item(),
            "mon_acc": mon_ok.float().mean().item(),
            "leap_acc": leap_ok.float().mean().item(),
            "decade_acc": decade_ok.float().mean().item()}

device     = "cuda" if torch.cuda.is_available() else "cpu"
cfg        = Model2Config()
train_cfg  = Train2Config()
path_cfg   = Path2Config()

os.makedirs(path_cfg.figures_dir, exist_ok=True)
os.makedirs(path_cfg.output_dir,  exist_ok=True)

print(f"Device: {device.upper()}")
train_ds, val_ds, _, _ = load_dataset(path_cfg.data_path, train_cfg.val_split, train_cfg.seed)
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}")

G, D, encoder, history = train_model2(train_ds, val_ds, cfg, train_cfg, device)

torch.save(G.state_dict(),       path_cfg.generator_path)
torch.save(D.state_dict(),       path_cfg.discriminator_path)
torch.save(encoder.state_dict(), path_cfg.encoder_path)
print(f"Weights saved → {path_cfg.output_dir}")

# Evaluate
from torch.utils.data import DataLoader
val_loader = DataLoader(val_ds, batch_size=512, shuffle=False)
all_X, all_Y = [], []
G.eval(); encoder.eval()
with torch.no_grad():
    for X_b, _ in val_loader:
        X_b = X_b.to(device)
        cond_emb = encoder(X_b)
        Y_b = G.sample(cond_emb, X_b, device=device).cpu()
        all_X.append(X_b.cpu()); all_Y.append(Y_b)
metrics = check_conditions(torch.cat(all_Y), torch.cat(all_X))
print("CSR:", metrics["csr"])

plot_loss_curves(history,         path_cfg.figures_dir)
plot_loss_log_scale(history,      path_cfg.figures_dir)
plot_condition_breakdown(metrics, path_cfg.figures_dir)

# Predict
X, _ = load_example_input(path_cfg.example_input_path)
X = X.to(device)
with torch.no_grad():
    cond_emb = encoder(X)
    Y_gen = G.sample(cond_emb, X, device=device).cpu()
with open(os.path.join(path_cfg.output_dir, "predictions.txt"), "w") as f:
    for i, cond in enumerate(X.cpu().tolist()):
        f.write(format_output_line(cond, Y_gen[i].tolist()) + "\\n")
print("Predictions saved.")
'''
with open('main2.py', 'w') as f:
    f.write(main2_src.strip())
print('✓ main2.py written')

In [ ]:
# ── 7. Create output directories ─────────────────────────────────────────────
import os
os.makedirs('output/figures', exist_ok=True)
print('✓ Directories ready')

In [ ]:
# ── 8. Run Model 2 ────────────────────────────────────────────────────────────
!python main2.py

In [ ]:
# ── 9. Show predictions ───────────────────────────────────────────────────────
print('First 10 predictions:')
with open('output/predictions.txt') as f:
    for i, line in enumerate(f):
        if i >= 10: break
        print(line, end='')

In [ ]:
# ── 10. Display figures ───────────────────────────────────────────────────────
from IPython.display import Image, display
import glob
figs = sorted(glob.glob('output/figures/*.png'))
print(f'Found {len(figs)} figures:')
for path in figs:
    print('\n', path)
    display(Image(path))

In [ ]:
# ── 11. Zip & download ────────────────────────────────────────────────────────
import zipfile, os
ZIP = 'output/model2_outputs.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in sorted(os.listdir('output/figures')):
        zf.write(f'output/figures/{fn}', f'figures/{fn}')
    if os.path.exists('output/predictions.txt'):
        zf.write('output/predictions.txt', 'predictions.txt')
    for name in ['generator.pt', 'discriminator.pt', 'encoder.pt']:
        p = f'output/{name}'
        if os.path.exists(p):
            zf.write(p, name)
print(f'✓ ZIP ({os.path.getsize(ZIP)/1e6:.1f} MB) → {ZIP}')
from google.colab import files
files.download(ZIP)